# Review 3 – Modelling & Final Application

## KSRTC EV Transition Analytics

### Project Objective

The objective of this project is to develop a machine learning based decision-support system for evaluating the potential transition of KSRTC bus depots from diesel to electric buses.

The system predicts the EV Suitability Score using historical operational data and supports future scenario analysis based on user-defined operational assumptions. It also estimates the economic, operational and environmental impact of EV adoption, including operating cost, OPEX savings, energy requirements, CO₂ reduction and transition economics.

### Review 3 Objectives

- Build and compare appropriate machine learning regression models.
- Perform hyperparameter tuning where applicable.
- Evaluate models using suitable regression metrics.
- Select and justify the best-performing model.
- Develop a future EV transition scenario engine.
- Estimate economic and environmental impacts.
- Rank KSRTC depots based on predicted EV suitability.
- Prepare the trained model for integration into the final TEJAS application.

In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import warnings
warnings.filterwarnings("ignore")

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
# Load the final preprocessed dataset from Review 2

processed_path = "../data/processed/TEJAS_PREPROCESSED.csv"

processed_df = pd.read_csv(processed_path)

print("Review 2 preprocessed dataset loaded successfully.")
print("Shape:", processed_df.shape)
print("Number of features:", processed_df.shape[1] - 1)
print("Target column:", "EV Suitability Score")

Review 2 preprocessed dataset loaded successfully.
Shape: (5520, 116)
Number of features: 115
Target column: EV Suitability Score


In [3]:
# Load original raw dataset for future scenario analysis

raw_path = "../data/raw/TEJAS_RAW_DATA.csv"

raw_df = pd.read_csv(raw_path)

print("Raw dataset loaded successfully.")
print("Shape:", raw_df.shape)
print("Number of depots:", raw_df["Depot ID"].nunique())
print("Years available:", sorted(raw_df["Year"].unique()))

Raw dataset loaded successfully.
Shape: (5520, 20)
Number of depots: 92
Years available: [np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025), np.int64(2026)]


In [4]:
# Define features and target

X = processed_df.drop(columns=["EV Suitability Score"])
y = processed_df["EV Suitability Score"]

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)

print("\nTarget variable:")
print("EV Suitability Score")

print("\nTarget statistics:")
print(y.describe())

Feature matrix shape: (5520, 115)
Target shape: (5520,)

Target variable:
EV Suitability Score

Target statistics:
count    5520.000000
mean        0.463777
std         0.116270
min         0.188500
25%         0.399000
50%         0.443500
75%         0.515200
max         0.776200
Name: EV Suitability Score, dtype: float64


In [5]:
# Time-based train/test split
# Train: 2021–2025
# Test: 2026 (partial-year validation)

train_mask = raw_df["Year"] < 2026
test_mask = raw_df["Year"] == 2026

X_train = X.loc[train_mask]
X_test = X.loc[test_mask]

y_train = y.loc[train_mask]
y_test = y.loc[test_mask]

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

print("\nTraining years:", sorted(raw_df.loc[train_mask, "Year"].unique()))
print("Testing years:", sorted(raw_df.loc[test_mask, "Year"].unique()))

Training samples: 5244
Testing samples: 276

Training years: [np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]
Testing years: [np.int64(2026)]


In [6]:
# Baseline Model — Linear Regression

linear_model = LinearRegression()

linear_model.fit(X_train, y_train)

linear_predictions = linear_model.predict(X_test)

print("Linear Regression trained successfully.")
print("Number of predictions:", len(linear_predictions))

Linear Regression trained successfully.
Number of predictions: 276


In [7]:
# Evaluate Linear Regression

linear_mae = mean_absolute_error(y_test, linear_predictions)
linear_rmse = np.sqrt(mean_squared_error(y_test, linear_predictions))
linear_r2 = r2_score(y_test, linear_predictions)

print("Linear Regression Performance")
print("-" * 35)
print(f"MAE  : {linear_mae:.4f}")
print(f"RMSE : {linear_rmse:.4f}")
print(f"R²   : {linear_r2:.4f}")

Linear Regression Performance
-----------------------------------
MAE  : 0.0141
RMSE : 0.0186
R²   : 0.7465


In [8]:
# Random Forest Regression

rf_model = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

rf_predictions = rf_model.predict(X_test)

print("Random Forest trained successfully.")
print("Number of predictions:", len(rf_predictions))

Random Forest trained successfully.
Number of predictions: 276


In [9]:
# Evaluate Random Forest

rf_mae = mean_absolute_error(y_test, rf_predictions)
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_predictions))
rf_r2 = r2_score(y_test, rf_predictions)

print("Random Forest Performance")
print("-" * 35)
print(f"MAE  : {rf_mae:.4f}")
print(f"RMSE : {rf_rmse:.4f}")
print(f"R²   : {rf_r2:.4f}")

Random Forest Performance
-----------------------------------
MAE  : 0.0002
RMSE : 0.0006
R²   : 0.9998


In [10]:
# Check Random Forest feature importance

feature_importance = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": rf_model.feature_importances_
}).sort_values("Importance", ascending=False)

print("Top 20 Random Forest Features:")
print(feature_importance.head(20).to_string(index=False))

Top 20 Random Forest Features:
                       Feature  Importance
               Buses Allocated    0.428254
   District_Thiruvananthapuram    0.178690
           Schedules Allocated    0.085631
            Passengers_per_Bus    0.068559
             District_Kottayam    0.051326
                          Year    0.029359
     Estimated EV Energy (MWh)    0.019111
        Estimated CO2 (Tonnes)    0.016487
       Estimated Diesel Litres    0.016424
               District_Kollam    0.015943
                  Effective KM    0.015762
Potential EV OPEX Saving (INR)    0.014611
                    Passengers    0.011924
       District_Pathanamthitta    0.008961
            District_Kozhikode    0.007608
             District_Palakkad    0.007584
             District_Thrissur    0.005594
           District_Malappuram    0.003290
            Depot ID_KSRTC-056    0.003103
            District_Kasaragod    0.002837


In [11]:
# Examine Random Forest prediction errors

rf_results = pd.DataFrame({
    "Actual": y_test.values,
    "Predicted": rf_predictions
})

rf_results["Absolute Error"] = np.abs(
    rf_results["Actual"] - rf_results["Predicted"]
)

print("Random Forest Prediction Error Summary")
print("-" * 45)
print(rf_results["Absolute Error"].describe())

print("\nLargest 10 prediction errors:")
print(
    rf_results.nlargest(10, "Absolute Error").to_string(index=False)
)

Random Forest Prediction Error Summary
---------------------------------------------
count    2.760000e+02
mean     1.802337e-04
std      5.226403e-04
min      5.551115e-17
25%      6.000000e-06
50%      3.025000e-05
75%      8.575000e-05
max      3.741000e-03
Name: Absolute Error, dtype: float64

Largest 10 prediction errors:
 Actual  Predicted  Absolute Error
 0.5265   0.522759        0.003741
 0.4227   0.419370        0.003330
 0.4516   0.454739        0.003139
 0.5236   0.520480        0.003120
 0.4494   0.446287        0.003113
 0.5549   0.551802        0.003098
 0.5027   0.504644        0.001944
 0.5202   0.518609        0.001591
 0.5554   0.554091        0.001309
 0.4197   0.418418        0.001282


In [12]:
# Gradient Boosting Regression

gb_model = GradientBoostingRegressor(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=3,
    random_state=42
)

gb_model.fit(X_train, y_train)

gb_predictions = gb_model.predict(X_test)

print("Gradient Boosting trained successfully.")
print("Number of predictions:", len(gb_predictions))

Gradient Boosting trained successfully.
Number of predictions: 276


In [13]:
# Evaluate Gradient Boosting

gb_mae = mean_absolute_error(y_test, gb_predictions)
gb_rmse = np.sqrt(mean_squared_error(y_test, gb_predictions))
gb_r2 = r2_score(y_test, gb_predictions)

print("Gradient Boosting Performance")
print("-" * 35)
print(f"MAE  : {gb_mae:.4f}")
print(f"RMSE : {gb_rmse:.4f}")
print(f"R²   : {gb_r2:.4f}")

Gradient Boosting Performance
-----------------------------------
MAE  : 0.0068
RMSE : 0.0098
R²   : 0.9292


In [14]:
# Compare baseline regression models

model_comparison = pd.DataFrame({
    "Model": [
        "Linear Regression",
        "Random Forest",
        "Gradient Boosting"
    ],
    "MAE": [
        linear_mae,
        rf_mae,
        gb_mae
    ],
    "RMSE": [
        linear_rmse,
        rf_rmse,
        gb_rmse
    ],
    "R2": [
        linear_r2,
        rf_r2,
        gb_r2
    ]
})

model_comparison = model_comparison.sort_values(
    by="R2",
    ascending=False
).reset_index(drop=True)

print(model_comparison.to_string(index=False))

            Model      MAE     RMSE       R2
    Random Forest 0.000180 0.000552 0.999776
Gradient Boosting 0.006795 0.009810 0.929243
Linear Regression 0.014065 0.018569 0.746477


In [15]:
# Check Random Forest training vs test performance

rf_train_predictions = rf_model.predict(X_train)

rf_train_r2 = r2_score(y_train, rf_train_predictions)
rf_test_r2 = r2_score(y_test, rf_predictions)

rf_train_mae = mean_absolute_error(y_train, rf_train_predictions)
rf_test_mae = mean_absolute_error(y_test, rf_predictions)

print("Random Forest — Training vs Test")
print("-" * 40)

print(f"Training MAE : {rf_train_mae:.6f}")
print(f"Testing MAE  : {rf_test_mae:.6f}")

print(f"\nTraining R²  : {rf_train_r2:.6f}")
print(f"Testing R²   : {rf_test_r2:.6f}")

Random Forest — Training vs Test
----------------------------------------
Training MAE : 0.001126
Testing MAE  : 0.000180

Training R²  : 0.999579
Testing R²   : 0.999776


In [16]:
# Core operational features for leakage/dependency check

core_features = [
    "Buses Allocated",
    "Schedules Allocated",
    "Effective KM",
    "Passengers",
    "Diesel Cost (INR/KM)",
    "EV Cost (INR/KM)",
    "EV Battery Capacity (kWh)",
    "EV Range (KM)",
    "Charging Power (kW)",
    "Electricity Tariff (INR/kWh)",
    "Year",
    "Month_Number",
    "Passengers_per_Bus"
]

# Keep only features that are available in the processed dataset
core_features = [col for col in core_features if col in processed_df.columns]

X_core = processed_df[core_features]

X_core_train = X_core.loc[train_mask]
X_core_test = X_core.loc[test_mask]

print("Core feature set created.")
print("Number of core features:", len(core_features))
print("\nFeatures:")
print(core_features)

Core feature set created.
Number of core features: 7

Features:
['Buses Allocated', 'Schedules Allocated', 'Effective KM', 'Passengers', 'Year', 'Month_Number', 'Passengers_per_Bus']


In [17]:
# Random Forest using core operational features only

rf_core_model = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

rf_core_model.fit(X_core_train, y_train)

rf_core_predictions = rf_core_model.predict(X_core_test)

rf_core_mae = mean_absolute_error(y_test, rf_core_predictions)
rf_core_rmse = np.sqrt(mean_squared_error(y_test, rf_core_predictions))
rf_core_r2 = r2_score(y_test, rf_core_predictions)

print("Core-Feature Random Forest Performance")
print("-" * 45)
print(f"MAE  : {rf_core_mae:.6f}")
print(f"RMSE : {rf_core_rmse:.6f}")
print(f"R²   : {rf_core_r2:.6f}")

Core-Feature Random Forest Performance
---------------------------------------------
MAE  : 0.000189
RMSE : 0.000918
R²   : 0.999380


In [18]:
# Correlation of core operational features with EV Suitability Score

core_analysis = X_core.copy()
core_analysis["EV Suitability Score"] = y.values

correlation = (
    core_analysis
    .corr(numeric_only=True)["EV Suitability Score"]
    .drop("EV Suitability Score")
    .sort_values(key=abs, ascending=False)
)

print("Correlation with EV Suitability Score")
print("-" * 45)
print(correlation)

Correlation with EV Suitability Score
---------------------------------------------
Passengers             0.682938
Buses Allocated        0.606243
Schedules Allocated    0.500606
Passengers_per_Bus     0.451638
Effective KM           0.282849
Year                  -0.149270
Month_Number           0.000244
Name: EV Suitability Score, dtype: float64


In [19]:
# Check training vs testing performance for the core-feature Random Forest

rf_core_train_predictions = rf_core_model.predict(X_core_train)

rf_core_train_r2 = r2_score(y_train, rf_core_train_predictions)
rf_core_test_r2 = r2_score(y_test, rf_core_predictions)

rf_core_train_mae = mean_absolute_error(
    y_train, rf_core_train_predictions
)
rf_core_test_mae = mean_absolute_error(
    y_test, rf_core_predictions
)

print("Core Random Forest — Training vs Test")
print("-" * 45)

print(f"Training MAE : {rf_core_train_mae:.6f}")
print(f"Testing MAE  : {rf_core_test_mae:.6f}")

print(f"\nTraining R²  : {rf_core_train_r2:.6f}")
print(f"Testing R²   : {rf_core_test_r2:.6f}")

Core Random Forest — Training vs Test
---------------------------------------------
Training MAE : 0.001289
Testing MAE  : 0.000189

Training R²  : 0.999164
Testing R²   : 0.999380


In [20]:
from sklearn.model_selection import RandomizedSearchCV

In [21]:
# Random Forest hyperparameter tuning

rf_param_grid = {
    "n_estimators": [100, 200, 300, 500],
    "max_depth": [None, 5, 10, 15, 20],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", "log2", 1.0]
}

rf_search = RandomizedSearchCV(
    estimator=RandomForestRegressor(
        random_state=42,
        n_jobs=-1
    ),
    param_distributions=rf_param_grid,
    n_iter=15,
    scoring="neg_root_mean_squared_error",
    cv=5,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

rf_search.fit(X_core_train, y_train)

print("Hyperparameter tuning completed.")
print("\nBest parameters:")
print(rf_search.best_params_)

print(f"\nBest CV RMSE: {-rf_search.best_score_:.6f}")

Fitting 5 folds for each of 15 candidates, totalling 75 fits
Hyperparameter tuning completed.

Best parameters:
{'n_estimators': 200, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 'log2', 'max_depth': 20}

Best CV RMSE: 0.020455


In [22]:
# Evaluate the best tuned Random Forest on the 2026 test set

best_rf_model = rf_search.best_estimator_

tuned_rf_predictions = best_rf_model.predict(X_core_test)

tuned_rf_mae = mean_absolute_error(y_test, tuned_rf_predictions)
tuned_rf_rmse = np.sqrt(mean_squared_error(y_test, tuned_rf_predictions))
tuned_rf_r2 = r2_score(y_test, tuned_rf_predictions)

print("Tuned Random Forest — 2026 Test Performance")
print("-" * 50)
print(f"MAE  : {tuned_rf_mae:.6f}")
print(f"RMSE : {tuned_rf_rmse:.6f}")
print(f"R²   : {tuned_rf_r2:.6f}")

Tuned Random Forest — 2026 Test Performance
--------------------------------------------------
MAE  : 0.000626
RMSE : 0.001835
R²   : 0.997524


In [23]:
# Final comparison of all evaluated models

final_comparison = pd.DataFrame({
    "Model": [
        "Linear Regression",
        "Gradient Boosting",
        "Random Forest",
        "Tuned Random Forest"
    ],
    "MAE": [
        linear_mae,
        gb_mae,
        rf_core_test_mae,
        tuned_rf_mae
    ],
    "RMSE": [
        linear_rmse,
        gb_rmse,
        rf_core_rmse,
        tuned_rf_rmse
    ],
    "R2": [
        linear_r2,
        gb_r2,
        rf_core_test_r2,
        tuned_rf_r2
    ]
})

final_comparison = final_comparison.sort_values(
    by="R2",
    ascending=False
).reset_index(drop=True)

print("Final Model Comparison — 2026 Test Set")
print("-" * 60)
print(final_comparison.to_string(index=False))

Final Model Comparison — 2026 Test Set
------------------------------------------------------------
              Model      MAE     RMSE       R2
      Random Forest 0.000189 0.000918 0.999380
Tuned Random Forest 0.000626 0.001835 0.997524
  Gradient Boosting 0.006795 0.009810 0.929243
  Linear Regression 0.014065 0.018569 0.746477


In [24]:
# Feature importance of the selected Random Forest model

final_feature_importance = pd.DataFrame({
    "Feature": X_core_train.columns,
    "Importance": rf_core_model.feature_importances_
}).sort_values(
    by="Importance",
    ascending=False
).reset_index(drop=True)

print("Final Random Forest Feature Importance")
print("-" * 50)
print(final_feature_importance.to_string(index=False))

Final Random Forest Feature Importance
--------------------------------------------------
            Feature  Importance
    Buses Allocated    0.461570
       Effective KM    0.219763
 Passengers_per_Bus    0.184750
Schedules Allocated    0.079117
         Passengers    0.049174
               Year    0.005082
       Month_Number    0.000546


In [25]:
# Train the final deployment model using all available historical data

final_model = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

final_model.fit(X_core, y)

print("Final deployment model trained successfully.")
print("Training samples:", len(X_core))
print("Features used:", X_core.shape[1])
print("Features:", list(X_core.columns))

Final deployment model trained successfully.
Training samples: 5520
Features used: 7
Features: ['Buses Allocated', 'Schedules Allocated', 'Effective KM', 'Passengers', 'Year', 'Month_Number', 'Passengers_per_Bus']


In [26]:
import joblib
import os

# Create models directory if it does not exist
os.makedirs("../models", exist_ok=True)

# Save final deployment model
model_path = "../models/tejas_ev_suitability_rf.pkl"
joblib.dump(final_model, model_path)

# Save exact feature order used by the model
feature_path = "../models/tejas_ev_features.pkl"
joblib.dump(list(X_core.columns), feature_path)

print("Final model saved successfully.")
print("Model:", model_path)
print("Feature list:", feature_path)

Final model saved successfully.
Model: ../models/tejas_ev_suitability_rf.pkl
Feature list: ../models/tejas_ev_features.pkl


In [27]:
# Create the latest available depot-level baseline

latest_year = raw_df["Year"].max()

latest_df = raw_df[
    raw_df["Year"] == latest_year
].copy()

latest_df = latest_df.sort_values(
    ["Depot ID", "Month"]
).reset_index(drop=True)

print("Latest baseline year:", latest_year)
print("Baseline rows:", len(latest_df))
print("Number of depots:", latest_df["Depot ID"].nunique())

print("\nBaseline columns:")
print(
    latest_df[
        [
            "Depot ID",
            "Depot Name",
            "District",
            "Month",
            "Buses Allocated",
            "Schedules Allocated",
            "Effective KM",
            "Passengers"
        ]
    ].head()
)

Latest baseline year: 2026
Baseline rows: 276
Number of depots: 92

Baseline columns:
    Depot ID Depot Name        District    Month  Buses Allocated  \
0  KSRTC-001      ADOOR  Pathanamthitta  2026-01               29   
1  KSRTC-001      ADOOR  Pathanamthitta  2026-02               28   
2  KSRTC-001      ADOOR  Pathanamthitta  2026-03               28   
3  KSRTC-002  ALAPPUZHA       Alappuzha  2026-01               67   
4  KSRTC-002  ALAPPUZHA       Alappuzha  2026-02               64   

   Schedules Allocated  Effective KM  Passengers  
0                   27        199640      271023  
1                   26        195127      264895  
2                   26        197169      267668  
3                   64        461396      626372  
4                   61        446444      606072  


In [29]:
# Create depot-level baseline using the latest available month

latest_month = latest_df["Month"].max()

depot_baseline = (
    latest_df[latest_df["Month"] == latest_month]
    [
        [
            "Depot ID",
            "Depot Name",
            "District",
            "Buses Allocated",
            "Schedules Allocated",
            "Effective KM",
            "Passengers"
        ]
    ]
    .copy()
    .reset_index(drop=True)
)

# Calculate the derived operational feature
depot_baseline["Passengers_per_Bus"] = (
    depot_baseline["Passengers"] /
    depot_baseline["Buses Allocated"]
)

depot_baseline["Year"] = latest_year
depot_baseline["Month_Number"] = 3

print("Depot-level baseline created.")
print("Baseline date:", latest_month)
print("Number of depots:", len(depot_baseline))
print("Baseline shape:", depot_baseline.shape)

print("\nFirst 5 depots:")
print(depot_baseline.head().to_string(index=False))

Depot-level baseline created.
Baseline date: 2026-03
Number of depots: 92
Baseline shape: (92, 10)

First 5 depots:
 Depot ID Depot Name           District  Buses Allocated  Schedules Allocated  Effective KM  Passengers  Passengers_per_Bus  Year  Month_Number
KSRTC-001      ADOOR     Pathanamthitta               28                   26        197169      267668         9559.571429  2026             3
KSRTC-002  ALAPPUZHA          Alappuzha               65                   61        448507      608873         9367.276923  2026             3
KSRTC-003      ALUVA          Ernakulam               48                   44        335537      455511         9489.812500  2026             3
KSRTC-004   ANKAMALY          Ernakulam               49                   46        342120      464448         9478.530612  2026             3
KSRTC-005   ATTINGAL Thiruvananthapuram               47                   59        323270      438858         9337.404255  2026             3


In [30]:
# Future EV transition scenario assumptions

future_year = 2027

bus_growth = 0.10
schedule_growth = 0.05
km_growth = 0.08
passenger_growth = 0.07

future_df = depot_baseline.copy()

future_df["Buses Allocated"] = (
    future_df["Buses Allocated"] * (1 + bus_growth)
).round().astype(int)

future_df["Schedules Allocated"] = (
    future_df["Schedules Allocated"] * (1 + schedule_growth)
).round().astype(int)

future_df["Effective KM"] = (
    future_df["Effective KM"] * (1 + km_growth)
).round().astype(int)

future_df["Passengers"] = (
    future_df["Passengers"] * (1 + passenger_growth)
).round().astype(int)

future_df["Passengers_per_Bus"] = (
    future_df["Passengers"] /
    future_df["Buses Allocated"]
)

future_df["Year"] = future_year
future_df["Month_Number"] = 3

print("Future scenario created successfully.")
print("Future year:", future_year)
print("Number of depots:", len(future_df))
print("Future scenario shape:", future_df.shape)

print("\nFirst 5 future depot scenarios:")
print(future_df.head().to_string(index=False))

Future scenario created successfully.
Future year: 2027
Number of depots: 92
Future scenario shape: (92, 10)

First 5 future depot scenarios:
 Depot ID Depot Name           District  Buses Allocated  Schedules Allocated  Effective KM  Passengers  Passengers_per_Bus  Year  Month_Number
KSRTC-001      ADOOR     Pathanamthitta               31                   27        212943      286405         9238.870968  2027             3
KSRTC-002  ALAPPUZHA          Alappuzha               72                   64        484388      651494         9048.527778  2027             3
KSRTC-003      ALUVA          Ernakulam               53                   46        362380      487397         9196.169811  2027             3
KSRTC-004   ANKAMALY          Ernakulam               54                   48        369490      496959         9202.944444  2027             3
KSRTC-005   ATTINGAL Thiruvananthapuram               52                   62        349132      469578         9030.346154  2027         

In [31]:
# Predict EV Suitability Score for the future scenario

future_X = future_df[
    [
        "Buses Allocated",
        "Schedules Allocated",
        "Effective KM",
        "Passengers",
        "Year",
        "Month_Number",
        "Passengers_per_Bus"
    ]
]

future_df["Predicted EV Suitability Score"] = final_model.predict(future_X)

# Keep the score within the valid 0–1 range
future_df["Predicted EV Suitability Score"] = (
    future_df["Predicted EV Suitability Score"].clip(0, 1)
)

print("Future EV suitability prediction completed.")
print(
    future_df[
        [
            "Depot ID",
            "Depot Name",
            "District",
            "Predicted EV Suitability Score"
        ]
    ].head(10).to_string(index=False)
)

print("\nScore statistics:")
print(
    future_df["Predicted EV Suitability Score"].describe()
)

Future EV suitability prediction completed.
 Depot ID      Depot Name           District  Predicted EV Suitability Score
KSRTC-001           ADOOR     Pathanamthitta                        0.399155
KSRTC-002       ALAPPUZHA          Alappuzha                        0.490112
KSRTC-003           ALUVA          Ernakulam                        0.416494
KSRTC-004        ANKAMALY          Ernakulam                        0.405018
KSRTC-005        ATTINGAL Thiruvananthapuram                        0.441330
KSRTC-006 CHADAYAMANGALAM             Kollam                        0.495928
KSRTC-007      CHALAKKUDY           Thrissur                        0.406370
KSRTC-008   CHANGANASSERY           Kottayam                        0.482681
KSRTC-009      CHATHANOOR             Kollam                        0.501329
KSRTC-010       CHENGANUR          Alappuzha                        0.416266

Score statistics:
count    92.000000
mean      0.435645
std       0.062775
min       0.281350
25%       0.40

In [32]:
# Rank depots by predicted EV suitability

depot_ranking = future_df[
    [
        "Depot ID",
        "Depot Name",
        "District",
        "Buses Allocated",
        "Schedules Allocated",
        "Effective KM",
        "Passengers",
        "Passengers_per_Bus",
        "Predicted EV Suitability Score"
    ]
].copy()

depot_ranking = depot_ranking.sort_values(
    "Predicted EV Suitability Score",
    ascending=False
).reset_index(drop=True)

depot_ranking["EV Priority Rank"] = (
    depot_ranking.index + 1
)

print("Depot ranking created successfully.")
print("\nTop 10 depots for EV transition:")
print(
    depot_ranking[
        [
            "EV Priority Rank",
            "Depot ID",
            "Depot Name",
            "District",
            "Predicted EV Suitability Score"
        ]
    ].head(10).to_string(index=False)
)

Depot ranking created successfully.

Top 10 depots for EV transition:
 EV Priority Rank  Depot ID      Depot Name  District  Predicted EV Suitability Score
                1 KSRTC-032          KOLLAM    Kollam                        0.559076
                2 KSRTC-024          KANNUR    Kannur                        0.558387
                3 KSRTC-074 SULTHAN BATHERY   Wayanad                        0.533127
                4 KSRTC-021        KALPETTA   Wayanad                        0.532980
                5 KSRTC-044    MANANTHAVADY   Wayanad                        0.531508
                6 KSRTC-037        KOTTAYAM  Kottayam                        0.529478
                7 KSRTC-026       KASARGODE Kasaragod                        0.519530
                8 KSRTC-022       KANGANGAD Kasaragod                        0.516249
                9 KSRTC-025  KARUNAGAPPALLY    Kollam                        0.508366
               10 KSRTC-063       PAYYANNUR    Kannur                 

In [33]:
# Check Thiruvananthapuram depot using the trained EV suitability model

tvm_df = future_df[
    future_df["Depot Name"].str.upper().str.contains("THIRUVANANTHAPURAM", na=False)
].copy()

print("Thiruvananthapuram depot records:")
print(
    tvm_df[
        [
            "Depot ID",
            "Depot Name",
            "District",
            "Buses Allocated",
            "Schedules Allocated",
            "Effective KM",
            "Passengers",
            "Passengers_per_Bus",
            "Predicted EV Suitability Score"
        ]
    ].to_string(index=False)
)

Thiruvananthapuram depot records:
Empty DataFrame
Columns: [Depot ID, Depot Name, District, Buses Allocated, Schedules Allocated, Effective KM, Passengers, Passengers_per_Bus, Predicted EV Suitability Score]
Index: []


In [34]:
# Find the actual Thiruvananthapuram depot name in the dataset

tvm_names = future_df[
    future_df["Depot Name"].astype(str).str.upper().str.contains(
        "THIRU|TRIV|TVM", regex=True, na=False
    )
][
    ["Depot ID", "Depot Name", "District"]
].drop_duplicates()

print(tvm_names.to_string(index=False))

 Depot ID           Depot Name           District
KSRTC-077           THIRUVALLA     Pathanamthitta
KSRTC-078         THIRUVAMBADY          Kozhikode
KSRTC-082 TRIVANDRUM - CENTRAL Thiruvananthapuram
KSRTC-083    TRIVANDRUM - CITY Thiruvananthapuram


In [35]:
tvm_df = future_df[
    future_df["Depot ID"].isin(["KSRTC-082", "KSRTC-083"])
].copy()

print(
    tvm_df[
        [
            "Depot ID",
            "Depot Name",
            "District",
            "Buses Allocated",
            "Schedules Allocated",
            "Effective KM",
            "Passengers",
            "Passengers_per_Bus",
            "Predicted EV Suitability Score"
        ]
    ].to_string(index=False)
)

 Depot ID           Depot Name           District  Buses Allocated  Schedules Allocated  Effective KM  Passengers  Passengers_per_Bus  Predicted EV Suitability Score
KSRTC-082 TRIVANDRUM - CENTRAL Thiruvananthapuram               70                   87        481516      647632         9251.885714                        0.493352
KSRTC-083    TRIVANDRUM - CITY Thiruvananthapuram               51                   62        346440      465958         9136.431373                        0.433003


In [36]:
# Rank Thiruvananthapuram depots among all 92 depots

ranking_check = future_df[
    [
        "Depot ID",
        "Depot Name",
        "District",
        "Predicted EV Suitability Score"
    ]
].copy()

ranking_check = ranking_check.sort_values(
    "Predicted EV Suitability Score",
    ascending=False
).reset_index(drop=True)

ranking_check["Rank"] = ranking_check.index + 1

tvm_ranking = ranking_check[
    ranking_check["Depot ID"].isin(["KSRTC-082", "KSRTC-083"])
]

print(tvm_ranking.to_string(index=False))

 Depot ID           Depot Name           District  Predicted EV Suitability Score  Rank
KSRTC-082 TRIVANDRUM - CENTRAL Thiruvananthapuram                        0.493352    17
KSRTC-083    TRIVANDRUM - CITY Thiruvananthapuram                        0.433003    47


In [37]:
# Top 10 depots by predicted EV suitability

top_ev_depots = depot_ranking[
    [
        "EV Priority Rank",
        "Depot ID",
        "Depot Name",
        "District",
        "Predicted EV Suitability Score"
    ]
].head(10)

print(top_ev_depots.to_string(index=False))

 EV Priority Rank  Depot ID      Depot Name  District  Predicted EV Suitability Score
                1 KSRTC-032          KOLLAM    Kollam                        0.559076
                2 KSRTC-024          KANNUR    Kannur                        0.558387
                3 KSRTC-074 SULTHAN BATHERY   Wayanad                        0.533127
                4 KSRTC-021        KALPETTA   Wayanad                        0.532980
                5 KSRTC-044    MANANTHAVADY   Wayanad                        0.531508
                6 KSRTC-037        KOTTAYAM  Kottayam                        0.529478
                7 KSRTC-026       KASARGODE Kasaragod                        0.519530
                8 KSRTC-022       KANGANGAD Kasaragod                        0.516249
                9 KSRTC-025  KARUNAGAPPALLY    Kollam                        0.508366
               10 KSRTC-063       PAYYANNUR    Kannur                        0.505251


In [38]:
# Export 2027 depot predictions for route analysis

import os

os.makedirs("../outputs", exist_ok=True)

future_df.to_csv(
    "../outputs/FUTURE_DEPOT_PREDICTIONS.csv",
    index=False
)

print("Future depot predictions exported successfully.")
print("Shape:", future_df.shape)
print("File: ../outputs/FUTURE_DEPOT_PREDICTIONS.csv")

Future depot predictions exported successfully.
Shape: (92, 11)
File: ../outputs/FUTURE_DEPOT_PREDICTIONS.csv
